# 05 数据预处理 ── 构建翻译数据流水线
原始文本 → 分词 → BPE编码 → 构建词表 → 转ID → 生成批次

In [ ]:
import torch, pickle
from bpe_utils import BPE, build_vocab
print('导入成功')


导入成功


In [ ]:
src = ['I love machine learning','Transformer is powerful','Attention is all you need','Deep learning is amazing']
for i,s in enumerate(src): print(f'句子{i+1}: {s}')


句子1: I love machine learning
句子2: Transformer is powerful
句子3: Attention is all you need
句子4: Deep learning is amazing


In [ ]:
tok = [t.lower().split() for t in src]
for i,t in enumerate(tok): print(f'句子{i+1}: {t}')


句子1: ['i', 'love', 'machine', 'learning']
句子2: ['transformer', 'is', 'powerful']
句子3: ['attention', 'is', 'all', 'you', 'need']
句子4: ['deep', 'learning', 'is', 'amazing']


In [ ]:
bpe = BPE.from_texts(src,20)
enc = [bpe.encode(t) for t in src]
print(f'合并规则: {len(bpe.merges)}')
for i,e in enumerate(enc): print(f'句子{i+1}: {e}')


合并规则: 20
句子1: ['i</w>', 'love</w>', 'mac', 'h', 'in', 'e</w>', 'learning</w>']
句子2: ['t', 'r', 'a', 'n', 's', 'f', 'o', 'r', 'm', 'er', '</w>', 'is</w>', 'p', 'o', 'w', 'er', 'f', 'u', 'l</w>']
句子3: ['a', 't', 't', 'e', 'n', 't', 'i', 'o', 'n', '</w>', 'is</w>', 'a', 'l', 'l</w>', 'y', 'o', 'u', '</w>', 'n', 'ee', 'd', '</w>']
句子4: ['d', 'ee', 'p', '</w>', 'learning</w>', 'is</w>', 'a', 'ma', 'z', 'ing</w>']


In [ ]:
vocab, id2tok = build_vocab(enc)
print(f'词表大小: {len(vocab)}')
for t,i in sorted(vocab.items(),key=lambda x:x[1])[:12]:
    print(f'  {i}: {repr(t)}')


词表大小: 35
  0: '<pad>'
  1: '<bos>'
  2: '<eos>'
  3: '<unk>'
  4: '</w>'
  5: 'a'
  6: 'd'
  7: 'e'
  8: 'e</w>'
  9: 'ee'
  10: 'er'
  11: 'f'


In [ ]:
def texts_to_ids(texts,bpe,vocab,max_len=20):
    ids = []
    for text in texts:
        t = bpe.encode(text)
        i = [vocab['<bos>']] + [vocab.get(x,vocab['<unk>']) for x in t] + [vocab['<eos>']]
        if len(i)>max_len: i = i[:max_len-1]+[vocab['<eos>']]
        else: i += [vocab['<pad>']]*(max_len-len(i))
        ids.append(i)
    return torch.tensor(ids)
ids = texts_to_ids(src,bpe,vocab)
print(f'形状: {list(ids.shape)}')
for i in range(len(src)): print(f'句子{i+1}: {ids[i].tolist()}')


形状: [4, 20]
句子1: [1, 14, 21, 24, 12, 15, 8, 20, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
句子2: [1, 30, 28, 5, 25, 29, 11, 26, 28, 22, 10, 4, 17, 27, 26, 32, 10, 11, 31, 2]
句子3: [1, 5, 30, 30, 7, 25, 30, 13, 26, 25, 4, 17, 5, 18, 19, 33, 26, 31, 4, 2]
句子4: [1, 6, 9, 27, 4, 20, 17, 5, 23, 34, 16, 2, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
def create_batches(data,bs=2):
    idx = torch.randperm(len(data))
    return [data[idx[i:i+bs]] for i in range(0,len(data),bs)]
batches = create_batches(ids,2)
print(f'{len(batches)} batches')
for i,b in enumerate(batches): print(f'  batch{i+1}: {list(b.shape)}')


2 batches
  batch1: [2, 20]
  batch2: [2, 20]


In [ ]:
data = {'ids':ids,'vocab':vocab,'id_to_token':id2tok,'bpe_merges':bpe.merges}
with open('preprocessed_data.pkl','wb') as f: pickle.dump(data,f)
print('已保存 preprocessed_data.pkl')
with open('preprocessed_data.pkl','rb') as f: ld = pickle.load(f)
print(f'验证: ids={list(ld["ids"].shape)}, vocab={len(ld["vocab"])}')


已保存 preprocessed_data.pkl
验证: ids=[4, 20], vocab=35


## 恭喜完成全部 5 个教程！
